**Part 4 — GenAI: Analyse patient follow-up notes**

In [1]:
!pip install -q -U google-genai pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [2]:
!pip install -q "google-genai==2.12.1" "google-auth==2.49.0" pydantic

In [1]:
import google
import google.genai
import google.auth
from importlib.metadata import version

print("Google GenAI version:", version("google-genai"))
print("Google Auth version:", version("google-auth"))
print("SDK import successful!")

Google GenAI version: 2.12.1
Google Auth version: 2.49.0
SDK import successful!


In [2]:
from google.colab import userdata
from google import genai

# Read API key from Colab Secrets
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

# Connect to Gemini
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini connection initialized successfully!")

Gemini connection initialized successfully!


In [3]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Reply with only the word OK."
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [4]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Reply with only the word OK."
)

print(response.text)

OK


In [5]:
import os

print("Files in current folder:")
print(os.listdir())

if os.path.exists("data"):
    print("\nFiles inside data folder:")
    print(os.listdir("data"))

Files in current folder:
['.config', 'sample_data']


In [6]:
import os

print(os.listdir())

['.config', 'followup_notes.csv', 'sample_data']


In [7]:
import pandas as pd

notes_df = pd.read_csv("followup_notes.csv")

print("Dataset shape:", notes_df.shape)

print("\nFirst 10 notes:")
display(notes_df.head(10))

Dataset shape: (30, 3)

First 10 notes:


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [8]:
notes_to_analyse = notes_df.head(10).copy()

print("Notes selected:", len(notes_to_analyse))

display(notes_to_analyse)

Notes selected: 10


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [9]:
allowed_root_causes = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

print("Allowed root causes:")

for cause in allowed_root_causes:
    print("-", cause)

Allowed root causes:
- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other


In [10]:
{
    "root_cause": "cost_or_fee",
    "confidence": 0.95,
    "summary": "Patient could not afford the consultation.",
    "recommended_action": "Offer a payment plan."
}

{'root_cause': 'cost_or_fee',
 'confidence': 0.95,
 'summary': 'Patient could not afford the consultation.',
 'recommended_action': 'Offer a payment plan.'}

In [11]:
from pydantic import BaseModel, Field

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)


print("Pydantic validation model created!")

Pydantic validation model created!


In [12]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {
            "type": "string"
        },
        "confidence": {
            "type": "number"
        },
        "summary": {
            "type": "string"
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

print("Gemini JSON schema created!")

Gemini JSON schema created!


In [13]:
import json
from google.genai import types

def analyse_note(note_text):

    system_instruction = """
    You are a careful healthcare follow-up note analyst.
    Classify patient notes accurately.
    Never invent information that is not present in the note.
    """

    prompt = f"""
    CONTEXT:
    CareLoop Clinics wants to understand why patients miss appointments.

    ROLE:
    Act as a professional healthcare data analyst.

    ACTION:
    Analyse the patient follow-up note and identify the main reason.

    Allowed root causes:
    {allowed_root_causes}

    If the reason is unclear, choose "other".
    Do not guess.

    TONE:
    Be professional, clear and concise.

    FORMAT:
    Return exactly these four fields:
    1. root_cause
    2. confidence (between 0 and 1)
    3. summary (maximum 25 words)
    4. recommended_action

    Patient note:
    {note_text}
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            max_output_tokens=1000,
            thinking_config=types.ThinkingConfig(
                thinking_level="MINIMAL"
            )
        )
    )

    result = json.loads(response.text)

    return result


print("analyse_note() function created successfully!")

analyse_note() function created successfully!


In [15]:
note_001 = notes_to_analyse.iloc[0]["note_text"]

print("Patient note:")
print(note_001)

print("\nGemini analysis:")

result_001 = analyse_note(note_001)

print(json.dumps(result_001, indent=4))

Patient note:
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

Gemini analysis:
{
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.0,
    "summary": "Patient missed the appointment because she did not receive a reminder message and forgot.",
    "recommended_action": "Reschedule the appointment for next week and verify contact information for reminder delivery."
}


In [16]:
{
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.95,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send appointment reminders."
}

{'root_cause': 'forgot_or_no_reminder',
 'confidence': 0.95,
 'summary': 'Patient forgot the appointment.',
 'recommended_action': 'Send appointment reminders.'}

**Task 4.1 — First Gemini analysis is complete**

In [17]:
from pydantic import ValidationError
import pandas as pd
import time

print("Validation tools ready!")

Validation tools ready!


In [18]:
def process_note(note_id, appointment_id, note_text):

    # Step 1: Handle blank notes without calling Gemini
    if pd.isna(note_text) or str(note_text).strip() == "":
        return {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        }

    # Step 2: Analyse the note using Gemini
    try:
        result = analyse_note(str(note_text))

        # Step 3: Validate the response using Pydantic
        validated = FollowupInsight(**result)

        # Step 4: Check whether the category is allowed
        if validated.root_cause not in allowed_root_causes:
            status = "invalid_category"
        else:
            status = "valid"

        return {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": validated.root_cause,
            "confidence": validated.confidence,
            "summary": validated.summary,
            "recommended_action": validated.recommended_action,
            "validation_status": status,
            "raw_response": json.dumps(result)
        }

    except Exception as error:
        print(f"Error processing {note_id}: {error}")

        return {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "",
            "confidence": None,
            "summary": "",
            "recommended_action": "",
            "validation_status": "error",
            "raw_response": ""
        }

print("process_note() function created successfully!")

process_note() function created successfully!


In [19]:
results = []

for index, row in notes_to_analyse.iterrows():

    print(f"Processing {row['note_id']}...")

    result = process_note(
        row["note_id"],
        row["appointment_id"],
        row["note_text"]
    )

    results.append(result)

    # Pause between API calls to respect free-tier limits
    if index != notes_to_analyse.index[-1]:
        time.sleep(13)

print("\nAll 10 notes processed!")

Processing N001...
Processing N002...
Processing N003...
Error processing N003: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing N004...
Processing N005...
Error processing N005: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing N006...
Processing N007...
Processing N008...
Error processing N008: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Processing N009...
Processing N010...
Error processing N010: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually t

In [20]:
results_df = pd.DataFrame(results)

display(results_df[
    ["note_id", "root_cause", "confidence", "validation_status"]
])

,note_id,root_cause,confidence,validation_status
0,N001,forgot_or_no_reminder,1.00,valid
1,N002,distance_or_transport,1.00,valid
2,N003,,NaN,error
3,N004,other,0.00,skipped_blank
4,N005,,NaN,error
5,N006,work_or_schedule,0.95,valid
6,N007,forgot_or_no_reminder,0.95,valid
7,N008,,NaN,error
8,N009,other,0.30,valid
9,N010,,NaN,error


In [21]:
print("Total rows:", len(results_df))

print("\nValidation status:")
print(results_df["validation_status"].value_counts())

print("\nMissing values:")
print(results_df.isnull().sum())

print("\nRoot cause distribution:")
print(results_df["root_cause"].value_counts())

Total rows: 10

Validation status:
validation_status
valid            5
error            4
skipped_blank    1
Name: count, dtype: int64

Missing values:
note_id               0
appointment_id        0
root_cause            0
confidence            4
summary               0
recommended_action    0
validation_status     0
raw_response          0
dtype: int64

Root cause distribution:
root_cause
                         4
forgot_or_no_reminder    2
other                    2
distance_or_transport    1
work_or_schedule         1
Name: count, dtype: int64


In [22]:
results_df.to_csv("notes_analysed.csv", index=False)

print("File saved successfully!")

print("\nColumns:")
print(results_df.columns.tolist())

print("\nShape:", results_df.shape)

File saved successfully!

Columns:
['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']

Shape: (10, 8)


**Task 4.3 (A) — Test invalid AI responses**

In [23]:
# Test 1: Confidence is invalid (1.7 instead of 0–1)
invalid_response_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot appointment",
    "recommended_action": "Send reminder"
}

# Test 2: Required field is missing
invalid_response_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.8,
    "summary": "Patient faced transport problems"
}

# Validate both responses
for number, response in enumerate(
    [invalid_response_1, invalid_response_2], start=1
):
    try:
        FollowupInsight(**response)
        print(f"Test {number}: Passed unexpectedly")

    except ValidationError as error:
        print(f"\nTest {number}: Invalid response detected!")
        print(error)


Test 1: Invalid response detected!
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Test 2: Invalid response detected!
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'distance_...ced transport problems'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


**Task 4.3(B) — Compare Prompt V1 vs V2**

In [27]:
def analyse_with_prompt(note_text, version):

    if version == "V1":
        prompt = f"""
        Classify this patient note.
        Return root_cause, confidence, summary,
        and recommended_action as JSON.

        Patient note: {note_text}
        """

    else:
        prompt = f"""
        CONTEXT:
        A clinic wants to understand why patients miss appointments.

        ROLE:
        Act as a careful healthcare data analyst.

        ACTION:
        Identify the main reason in the patient note.

        Allowed root causes:
        {allowed_root_causes}

        If the reason is unclear, choose "other".
        Never guess or invent information.

        FORMAT:
        Return root_cause, confidence (0 to 1),
        summary (maximum 25 words), and recommended_action.

        Patient note: {note_text}
        """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            max_output_tokens=1000
        )
    )

    return json.loads(response.text)


print("V1 and V2 prompt functions created!")

V1 and V2 prompt functions created!


In [28]:
comparison_results = []

for note_id in ["N007", "N009"]:

    note_text = notes_to_analyse.loc[
        notes_to_analyse["note_id"] == note_id,
        "note_text"
    ].iloc[0]

    for version in ["V1", "V2"]:

        print(f"Analysing {note_id} with {version}...")

        try:
            output = analyse_with_prompt(note_text, version)

            comparison_results.append({
                "note_id": note_id,
                "prompt_version": version,
                "root_cause": output["root_cause"],
                "confidence": output["confidence"],
                "is_allowed_category": (
                    output["root_cause"] in allowed_root_causes
                )
            })

            print("Success!")

        except Exception as error:
            print("API error:", error)

            comparison_results.append({
                "note_id": note_id,
                "prompt_version": version,
                "root_cause": "",
                "confidence": None,
                "is_allowed_category": False
            })

        time.sleep(20)

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

Analysing N007 with V1...
API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Analysing N007 with V2...
API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Analysing N009 with V1...
API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Analysing N009 with V2...
API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


,note_id,prompt_version,root_cause,confidence,is_allowed_category
0,N007,V1,,None,False
1,N007,V2,,None,False
2,N009,V1,,None,False
3,N009,V2,,None,False


In [29]:
v2_existing = results_df[
    results_df["note_id"].isin(["N007", "N009"])
][["note_id", "root_cause", "confidence", "validation_status"]]

display(v2_existing)

,note_id,root_cause,confidence,validation_status
6,N007,forgot_or_no_reminder,0.95,valid
8,N009,other,0.30,valid


**Task 4.3(C) — Count API calls**

In [30]:
print("Task 4.1: 1 API call")
print("Task 4.2: 9 API calls")
print("Task 4.3(B): 0 successful calls (4 attempts failed)")

print("\nTotal successful API calls:", 10)
print("Total failed attempts:", 4)
print("Total attempts:", 14)

Task 4.1: 1 API call
Task 4.2: 9 API calls
Task 4.3(B): 0 successful calls (4 attempts failed)

Total successful API calls: 10
Total failed attempts: 4
Total attempts: 14


**Task 4.4 — Connect AI findings with appointment data**

In [32]:
appointments_df = pd.read_csv("appointments_clean.csv")

merged_df = results_df.merge(
    appointments_df[["appointment_id", "reminder_sent", "no_show"]],
    on="appointment_id",
    how="left"
)

print("Merged dataset shape:", merged_df.shape)

display(
    merged_df[
        ["note_id", "appointment_id", "root_cause",
         "reminder_sent", "no_show"]
    ]
)

Merged dataset shape: (10, 10)


,note_id,appointment_id,root_cause,reminder_sent,no_show
0,N001,A0872,forgot_or_no_reminder,No,1
1,N002,A0897,distance_or_transport,No,1
2,N003,A0526,,Yes,1
3,N004,A0926,other,No,1
4,N005,A0534,,Yes,1
5,N006,A0507,work_or_schedule,Yes,1
6,N007,A0333,forgot_or_no_reminder,No,1
7,N008,A0222,,Yes,1
8,N009,A0977,other,Yes,1
9,N010,A0596,,No,1


In [33]:
# Keep original merged data unchanged
analysis_df = merged_df.copy()

# Label missing AI classifications honestly
analysis_df["root_cause"] = (
    analysis_df["root_cause"].fillna("").replace(
        "", "unclassified_or_error"
    )
)

# Compare AI root causes with reminder status
cause_reminder_table = pd.crosstab(
    analysis_df["root_cause"],
    analysis_df["reminder_sent"]
)

display(cause_reminder_table)

reminder_sent,No,Yes
root_cause,,
distance_or_transport,1,0
forgot_or_no_reminder,2,0
other,1,1
unclassified_or_error,1,3
work_or_schedule,0,1


In [34]:
print("Notes analysed:", len(results_df))
print("Merged appointments:", len(merged_df))
print("Output columns:", results_df.columns.tolist())

print("\nValidation status:")
print(results_df["validation_status"].value_counts())

print("\nSaved files:")
import os
print("notes_analysed.csv exists:", os.path.exists("notes_analysed.csv"))

Notes analysed: 10
Merged appointments: 10
Output columns: ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']

Validation status:
validation_status
valid            5
error            4
skipped_blank    1
Name: count, dtype: int64

Saved files:
notes_analysed.csv exists: True
